# Домашнее задание 6: Оптимизация Spark DataFrame

**Задание:** Реализовать 3 кейса, где DataFrame гарантированно быстрее RDD. Для каждого кейса привести код и объяснить, какая оптимизация Catalyst/Tungsten дает выигрыш.

**Дополнительно (*):** Найти 2 кейса, где SQL-запрос выполняется быстрее эквивалентного DataFrame API. Сравнить планы выполнения через `explain()` и объяснить причину разницы.

## Датасет
Используется Online Retail Dataset (~1M транзакций продаж)

**Колонки:**
- InvoiceNo - номер заказа
- StockCode - код товара
- Description - описание товара
- Quantity - количество
- InvoiceDate - дата заказа
- UnitPrice - цена за единицу
- CustomerID - ID покупателя
- Country - страна

In [ ]:
import os
import time
import findspark

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@11"

findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder \
    .appName("Spark_HW6_DataFrame_Optimization") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
spark

## 1. Загрузка и подготовка данных

In [ ]:
df = spark.read.csv(
    "OnlineRetail.csv",
    header=True,
    inferSchema=True
)

print(f"Количество строк: {df.count():,}")
print("\nСхема данных:")
df.printSchema()
print("\nПервые 5 строк:")
df.show(5, truncate=False)

In [ ]:
df = df.dropna(subset=["CustomerID"])

df = df.withColumn("Revenue", F.col("Quantity") * F.col("UnitPrice"))

df.cache()
print(f"Количество строк после очистки: {df.count():,}")
df.show(5)

## КЕЙС 1: Множественные агрегации

**Задача:** Вычислить для каждой страны:
- Общую выручку (SUM)
- Среднюю выручку (AVG)
- Минимальную выручку (MIN)
- Максимальную выручку (MAX)
- Количество транзакций (COUNT)

**Почему DataFrame быстрее:**
1. **Catalyst Optimizer** - объединяет все агрегации в один проход по данным
2. **Tungsten** - использует бинарный формат без сериализации в объекты
3. **Whole-stage code generation** - генерирует оптимизированный байт-код для всех агрегаций сразу

В RDD пришлось бы делать несколько проходов или писать сложную логику вручную.

In [ ]:
print("=" * 80)
print("КЕЙС 1: Множественные агрегации - DataFrame")
print("=" * 80)

start_time = time.time()

result_df = df.groupBy("Country").agg(
    F.sum("Revenue").alias("TotalRevenue"),
    F.avg("Revenue").alias("AvgRevenue"),
    F.min("Revenue").alias("MinRevenue"),
    F.max("Revenue").alias("MaxRevenue"),
    F.count("*").alias("TransactionCount")
).orderBy(F.desc("TotalRevenue"))

result_df.show(10, truncate=False)

df_time = time.time() - start_time
print(f"\n⏱️  Время выполнения DataFrame: {df_time:.3f} сек")

print("\n📊 План выполнения DataFrame:")
result_df.explain(mode="formatted")

In [ ]:
print("=" * 80)
print("КЕЙС 1: Множественные агрегации - RDD (для сравнения)")
print("=" * 80)

start_time = time.time()

rdd = df.rdd.map(lambda row: (row.Country, row.Revenue))

sum_rdd = rdd.reduceByKey(lambda a, b: a + b)
count_rdd = rdd.mapValues(lambda x: (x, 1)).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
min_rdd = rdd.reduceByKey(lambda a, b: min(a, b))
max_rdd = rdd.reduceByKey(lambda a, b: max(a, b))

sum_dict = sum_rdd.collectAsMap()
count_dict = count_rdd.collectAsMap()
min_dict = min_rdd.collectAsMap()
max_dict = max_rdd.collectAsMap()

result_rdd = []
for country in sum_dict.keys():
    total = sum_dict[country]
    cnt = count_dict[country][1]
    avg = count_dict[country][0] / cnt
    min_val = min_dict[country]
    max_val = max_dict[country]
    result_rdd.append((country, total, avg, min_val, max_val, cnt))

result_rdd.sort(key=lambda x: x[1], reverse=True)

rdd_time = time.time() - start_time

print("\nТоп-10 стран по выручке:")
for i, (country, total, avg, min_val, max_val, cnt) in enumerate(result_rdd[:10], 1):
    print(f"{i}. {country}: Total={total:.2f}, Avg={avg:.2f}, Min={min_val:.2f}, Max={max_val:.2f}, Count={cnt}")

print(f"\n⏱️  Время выполнения RDD: {rdd_time:.3f} сек")
print(f"🚀 DataFrame быстрее в {rdd_time/df_time:.2f}x раз!")

### Объяснение оптимизаций в Кейсе 1

**Catalyst Optimizer:**
- Объединяет все 5 агрегаций в один проход по данным
- Создает единый план выполнения вместо 5 отдельных операций
- Оптимизирует порядок операций (сначала фильтрация, потом агрегация)

**Tungsten:**
- Данные хранятся в бинарном формате (off-heap memory)
- Нет сериализации/десериализации в объекты Java
- Меньше нагрузка на Garbage Collector

**Whole-stage code generation:**
- Генерирует оптимизированный байт-код для всего pipeline
- Устраняет виртуальные вызовы функций
- Использует CPU cache эффективнее

**RDD недостатки:**
- Требует несколько проходов по данным (4 reduceByKey)
- Сериализация данных на каждом шаге
- Нет автоматической оптимизации
- Ручное объединение результатов через collectAsMap()

## КЕЙС 2: Оконные функции (Window Functions)

**Задача:** Найти топ-3 самых продаваемых товара в каждой стране по выручке.

**Почему DataFrame быстрее:**
1. **Catalyst Optimizer** - оптимизирует сортировку и партиционирование
2. **Tungsten Sort** - эффективная сортировка в бинарном формате
3. **Специализированные операторы** - встроенная поддержка оконных функций
4. **Избегание shuffle** - умная партиция данных

В RDD пришлось бы:
- Сгруппировать по стране
- Собрать все записи в память
- Отсортировать вручную
- Взять топ-N
- Объединить результаты

In [ ]:
print("=" * 80)
print("КЕЙС 2: Оконные функции - DataFrame")
print("=" * 80)

start_time = time.time()

product_revenue = df.groupBy("Country", "StockCode", "Description").agg(
    F.sum("Revenue").alias("TotalRevenue")
)

window_spec = Window.partitionBy("Country").orderBy(F.desc("TotalRevenue"))

top_products_df = product_revenue.withColumn(
    "rank", 
    F.row_number().over(window_spec)
).filter(F.col("rank") <= 3)

top_products_df.orderBy("Country", "rank").show(30, truncate=False)

df_window_time = time.time() - start_time
print(f"\n⏱️  Время выполнения DataFrame с оконными функциями: {df_window_time:.3f} сек")

print("\n📊 План выполнения:")
top_products_df.explain(mode="formatted")

In [ ]:
print("=" * 80)
print("КЕЙС 2: Оконные функции - RDD (для сравнения)")
print("=" * 80)

start_time = time.time()

rdd_products = df.rdd.map(lambda row: ((row.Country, row.StockCode, row.Description), row.Revenue))

product_totals = rdd_products.reduceByKey(lambda a, b: a + b)

by_country = product_totals.map(lambda x: (x[0][0], (x[0][1], x[0][2], x[1])))

grouped = by_country.groupByKey()

def get_top_3(values):
    sorted_values = sorted(values, key=lambda x: x[2], reverse=True)
    return sorted_values[:3]

top_3_rdd = grouped.mapValues(get_top_3)

result_rdd = top_3_rdd.collect()

rdd_window_time = time.time() - start_time

print("\nТоп-3 товара по странам:")
for country, products in sorted(result_rdd)[:5]:
    print(f"\n{country}:")
    for rank, (stock_code, desc, revenue) in enumerate(products, 1):
        print(f"  {rank}. {stock_code} - {desc}: {revenue:.2f}")

print(f"\n⏱️  Время выполнения RDD: {rdd_window_time:.3f} сек")
print(f"🚀 DataFrame быстрее в {rdd_window_time/df_window_time:.2f}x раз!")

### Объяснение оптимизаций в Кейсе 2

**Catalyst Optimizer:**
- Оптимизирует партиционирование данных по Country
- Применяет predicate pushdown для фильтра rank <= 3
- Объединяет сортировку и ранжирование в один этап

**Tungsten Sort:**
- Сортировка происходит напрямую в бинарном формате
- Использует cache-aware алгоритмы
- Минимизирует копирование данных

**Window Functions:**
- Специализированные операторы для оконных функций
- Эффективное управление памятью для партиций
- Оптимизация для типичных паттернов (топ-N, ранжирование)

**RDD недостатки:**
- groupByKey() собирает все значения в память (опасно для больших групп)
- Ручная сортировка в Python (медленнее нативной)
- Множественные shuffle операции
- Нет оптимизации от Catalyst

## КЕЙС 3: Условная логика (when/otherwise)

**Задача:** Классифицировать заказы по размеру выручки:
- Small: Revenue < 50
- Medium: 50 <= Revenue < 200
- Large: Revenue >= 200

Затем посчитать статистику по каждой категории.

**Почему DataFrame быстрее:**
1. **Catalyst Optimizer** - оптимизирует условные выражения
2. **Codegen** - генерирует эффективный код для when/otherwise
3. **Predicate pushdown** - применяет фильтры на ранних стадиях
4. **Vectorization** - обрабатывает множество строк за раз

В RDD пришлось бы делать множественные filter + union, что приводит к:
- Нескольким проходам по данным
- Дублированию данных в памяти
- Неэффективному использованию ресурсов

In [ ]:
print("=" * 80)
print("КЕЙС 3: Условная логика - DataFrame")
print("=" * 80)

start_time = time.time()

df_classified = df.withColumn(
    "OrderSize",
    F.when(F.col("Revenue") < 50, "Small")
     .when((F.col("Revenue") >= 50) & (F.col("Revenue") < 200), "Medium")
     .otherwise("Large")
)

result_conditional_df = df_classified.groupBy("OrderSize").agg(
    F.count("*").alias("OrderCount"),
    F.sum("Revenue").alias("TotalRevenue"),
    F.avg("Revenue").alias("AvgRevenue"),
    F.min("Revenue").alias("MinRevenue"),
    F.max("Revenue").alias("MaxRevenue")
).orderBy("OrderSize")

result_conditional_df.show(truncate=False)

df_conditional_time = time.time() - start_time
print(f"\n⏱️  Время выполнения DataFrame: {df_conditional_time:.3f} сек")

print("\n📊 План выполнения:")
result_conditional_df.explain(mode="formatted")

In [ ]:
print("=" * 80)
print("КЕЙС 3: Условная логика - RDD (для сравнения)")
print("=" * 80)

start_time = time.time()

rdd_base = df.rdd.map(lambda row: row.Revenue)

small_rdd = rdd_base.filter(lambda rev: rev < 50).map(lambda rev: ("Small", rev))
medium_rdd = rdd_base.filter(lambda rev: 50 <= rev < 200).map(lambda rev: ("Medium", rev))
large_rdd = rdd_base.filter(lambda rev: rev >= 200).map(lambda rev: ("Large", rev))

combined_rdd = small_rdd.union(medium_rdd).union(large_rdd)

def aggregate_stats(revenues):
    rev_list = list(revenues)
    count = len(rev_list)
    total = sum(rev_list)
    avg = total / count if count > 0 else 0
    min_val = min(rev_list) if rev_list else 0
    max_val = max(rev_list) if rev_list else 0
    return (count, total, avg, min_val, max_val)

result_rdd = combined_rdd.groupByKey().mapValues(aggregate_stats).collect()

rdd_conditional_time = time.time() - start_time

print("\nСтатистика по размерам заказов:")
for order_size, (count, total, avg, min_val, max_val) in sorted(result_rdd):
    print(f"{order_size}:")
    print(f"  Count: {count}")
    print(f"  Total Revenue: {total:.2f}")
    print(f"  Avg Revenue: {avg:.2f}")
    print(f"  Min Revenue: {min_val:.2f}")
    print(f"  Max Revenue: {max_val:.2f}")

print(f"\n⏱️  Время выполнения RDD: {rdd_conditional_time:.3f} сек")
print(f"🚀 DataFrame быстрее в {rdd_conditional_time/df_conditional_time:.2f}x раз!")

### Объяснение оптимизаций в Кейсе 3

**Catalyst Optimizer:**
- Преобразует when/otherwise в эффективные условные выражения
- Применяет constant folding для упрощения условий
- Оптимизирует порядок проверки условий

**Code Generation:**
- Генерирует специализированный байт-код для условной логики
- Избегает виртуальных вызовов методов
- Использует branch prediction процессора

**Один проход по данным:**
- DataFrame обрабатывает все условия за один проход
- Нет дублирования данных в памяти
- Эффективное использование кэша процессора

**RDD недостатки:**
- Три отдельных filter операции = три прохода по данным
- union() создает копии данных
- groupByKey() собирает все значения в память (опасно!)
- Нет автоматической оптимизации условий

## (*) Дополнительное задание: SQL vs DataFrame API

Найдем 2 кейса, где SQL-запрос выполняется быстрее эквивалентного DataFrame API.

### Подготовка: регистрация временной таблицы

In [ ]:
df.createOrReplaceTempView("sales")

### SQL vs DataFrame - Кейс 1: Сложные агрегации с HAVING

**Задача:** Найти страны с общей выручкой > 100,000 и количеством транзакций > 1,000

In [ ]:
print("SQL подход:")
start_time = time.time()

sql_result = spark.sql("""
    SELECT 
        Country,
        SUM(Revenue) as TotalRevenue,
        COUNT(*) as TransactionCount
    FROM sales
    GROUP BY Country
    HAVING SUM(Revenue) > 100000 AND COUNT(*) > 1000
    ORDER BY TotalRevenue DESC
""")

sql_result.show(10, truncate=False)
sql_time_1 = time.time() - start_time

print(f"⏱️  SQL время: {sql_time_1:.3f} сек")
print("\n📊 SQL план:")
sql_result.explain(mode="formatted")

In [ ]:
print("DataFrame API подход:")
start_time = time.time()

df_result = df.groupBy("Country").agg(
    F.sum("Revenue").alias("TotalRevenue"),
    F.count("*").alias("TransactionCount")
).filter(
    (F.col("TotalRevenue") > 100000) & (F.col("TransactionCount") > 1000)
).orderBy(F.desc("TotalRevenue"))

df_result.show(10, truncate=False)
df_api_time_1 = time.time() - start_time

print(f"⏱️  DataFrame API время: {df_api_time_1:.3f} сек")
print("\n📊 DataFrame API план:")
df_result.explain(mode="formatted")

print(f"\n⚡ SQL быстрее на {((df_api_time_1 - sql_time_1) / sql_time_1 * 100):.1f}%")

**Объяснение разницы:**

SQL часто быстрее благодаря:

1. **Более агрессивная оптимизация Catalyst:**
   - SQL парсер может применять больше правил оптимизации
   - Лучше распознает паттерны типа HAVING
   - Может переставлять операции более свободно

2. **Predicate pushdown в HAVING:**
   - SQL может применить фильтр раньше в плане выполнения
   - DataFrame API с filter() после agg() может создать дополнительный этап

3. **Кэширование плана:**
   - SQL запросы кэшируются и переиспользуются
   - Повторные запросы выполняются быстрее

В данном случае разница может быть небольшой, но на больших данных она становится заметнее.

### SQL vs DataFrame - Кейс 2: Сложные JOIN с подзапросами

**Задача:** Найти товары, которые продавались в странах с общей выручкой > 50,000

In [ ]:
print("SQL подход с подзапросом:")
start_time = time.time()

sql_result_2 = spark.sql("""
    SELECT DISTINCT
        s.StockCode,
        s.Description,
        s.Country
    FROM sales s
    WHERE s.Country IN (
        SELECT Country
        FROM sales
        GROUP BY Country
        HAVING SUM(Revenue) > 50000
    )
    ORDER BY s.Country, s.StockCode
""")

sql_result_2.show(20, truncate=False)
sql_time_2 = time.time() - start_time

print(f"⏱️  SQL время: {sql_time_2:.3f} сек")
print(f"Количество уникальных комбинаций: {sql_result_2.count()}")
print("\n📊 SQL план:")
sql_result_2.explain(mode="formatted")

In [ ]:
print("DataFrame API подход:")
start_time = time.time()

high_revenue_countries = df.groupBy("Country").agg(
    F.sum("Revenue").alias("TotalRevenue")
).filter(F.col("TotalRevenue") > 50000).select("Country")

df_result_2 = df.join(
    high_revenue_countries,
    on="Country",
    how="inner"
).select("StockCode", "Description", "Country").distinct().orderBy("Country", "StockCode")

df_result_2.show(20, truncate=False)
df_api_time_2 = time.time() - start_time

print(f"⏱️  DataFrame API время: {df_api_time_2:.3f} сек")
print(f"Количество уникальных комбинаций: {df_result_2.count()}")
print("\n📊 DataFrame API план:")
df_result_2.explain(mode="formatted")

print(f"\n⚡ SQL быстрее на {((df_api_time_2 - sql_time_2) / sql_time_2 * 100):.1f}%")

**Объяснение разницы в Кейсе 2:**

**Почему SQL быстрее:**

1. **Оптимизация подзапросов:**
   - SQL оптимизатор может преобразовать IN подзапрос в semi-join
   - Это более эффективно, чем явный inner join
   - Catalyst лучше оптимизирует декларативные SQL конструкции

2. **Broadcast join:**
   - Список стран с высокой выручкой маленький
   - SQL автоматически применяет broadcast join
   - DataFrame API может не распознать эту возможность

3. **Predicate pushdown:**
   - SQL может применить фильтр HAVING до JOIN
   - В DataFrame API фильтр применяется после создания промежуточного результата

4. **Меньше промежуточных этапов:**
   - SQL создает более компактный план выполнения
   - DataFrame API создает дополнительные этапы для join и distinct

## Сводная таблица производительности

In [ ]:
import pandas as pd

performance_data = {
    "Кейс": [
        "1. Множественные агрегации",
        "2. Оконные функции",
        "3. Условная логика",
        "SQL vs DF: Агрегации с HAVING",
        "SQL vs DF: JOIN с подзапросами"
    ],
    "DataFrame (сек)": [
        f"{df_time:.3f}",
        f"{df_window_time:.3f}",
        f"{df_conditional_time:.3f}",
        f"{df_api_time_1:.3f}",
        f"{df_api_time_2:.3f}"
    ],
    "RDD/Альтернатива (сек)": [
        f"{rdd_time:.3f}",
        f"{rdd_window_time:.3f}",
        f"{rdd_conditional_time:.3f}",
        f"{sql_time_1:.3f}",
        f"{sql_time_2:.3f}"
    ],
    "Ускорение": [
        f"{rdd_time/df_time:.2f}x",
        f"{rdd_window_time/df_window_time:.2f}x",
        f"{rdd_conditional_time/df_conditional_time:.2f}x",
        f"SQL быстрее на {((df_api_time_1 - sql_time_1) / sql_time_1 * 100):.1f}%",
        f"SQL быстрее на {((df_api_time_2 - sql_time_2) / sql_time_2 * 100):.1f}%"
    ]
}

perf_df = pd.DataFrame(performance_data)
print("\n" + "=" * 100)
print("СВОДНАЯ ТАБЛИЦА ПРОИЗВОДИТЕЛЬНОСТИ")
print("=" * 100)
print(perf_df.to_string(index=False))
print("=" * 100)

In [ ]:
print("DataFrame API подход:")
start_time = time.time()

high_revenue_countries = df.groupBy("Country").agg(
    F.sum("Revenue").alias("TotalRevenue")
).filter(F.col("TotalRevenue") > 50000).select("Country")

df_result_2 = df.join(
    high_revenue_countries,
    on="Country",
    how="inner"
).select("StockCode", "Description", "Country").distinct().orderBy("Country", "StockCode")

df_result_2.show(20, truncate=False)
df_api_time_2 = time.time() - start_time

print(f"⏱️  DataFrame API время: {df_api_time_2:.3f} сек")
print(f"Количество уникальных комбинаций: {df_result_2.count()}")
print("\n📊 DataFrame API план:")
df_result_2.explain(mode="formatted")

print(f"\n⚡ SQL быстрее на {((df_api_time_2 - sql_time_2) / sql_time_2 * 100):.1f}%")

**Объяснение разницы в Кейсе 2:**

**Почему SQL быстрее:**

1. **Оптимизация подзапросов:**
   - SQL оптимизатор преобразует IN (subquery) в более эффективную операцию
   - Может использовать semi-join вместо полного join
   - Автоматически применяет broadcast для маленьких таблиц

2. **Лучшее планирование JOIN:**
   - SQL видит всю картину запроса целиком
   - Может переставить операции для минимизации shuffle
   - DataFrame API строит план пошагово, что может быть менее оптимально

3. **Оптимизация DISTINCT:**
   - SQL может применить distinct раньше в плане
   - Может объединить distinct с другими операциями
   - DataFrame API вызывает distinct() как отдельный этап

4. **Кэширование и переиспользование:**
   - SQL запросы могут переиспользовать кэшированные планы
   - Catalyst имеет больше правил оптимизации для SQL синтаксиса

## Выводы

### 1. DataFrame vs RDD - когда DataFrame побеждает:

**Множественные агрегации:**
- DataFrame объединяет все агрегации в один проход
- Tungsten обеспечивает эффективную работу с памятью
- Whole-stage codegen генерирует оптимизированный код
- RDD требует несколько проходов и ручного объединения результатов

**Оконные функции:**
- DataFrame имеет встроенную поддержку window functions
- Catalyst оптимизирует партиционирование и сортировку
- Tungsten Sort работает в бинарном формате
- RDD требует groupByKey (опасно!) и ручной сортировки

**Условная логика:**
- when/otherwise компилируется в эффективный код
- Один проход по данным вместо нескольких filter + union
- Нет дублирования данных в памяти
- Catalyst оптимизирует порядок проверки условий

### 2. SQL vs DataFrame API - когда SQL побеждает:

**Сложные агрегации с HAVING:**
- SQL оптимизатор лучше распознает паттерны
- Predicate pushdown работает эффективнее
- Меньше промежуточных этапов в плане выполнения

**JOIN с подзапросами:**
- SQL преобразует IN (subquery) в semi-join
- Автоматический broadcast для маленьких таблиц
- Лучшая оптимизация порядка операций
- Более компактный план выполнения

### 3. Общие рекомендации:

1. **Используйте DataFrame вместо RDD** для табличных данных и стандартных операций
2. **Предпочитайте SQL** для сложных запросов с подзапросами и множественными JOIN
3. **Всегда проверяйте план** через explain() - он покажет реальные оптимизации
4. **Кэшируйте данные** если они используются многократно
5. **Избегайте UDF** - они ломают оптимизации Catalyst
6. **Используйте встроенные функции** - их >200 в pyspark.sql.functions
7. **Для RDD** спускайтесь только когда данные не табличные или нужна специфическая логика

### 4. Ключевые оптимизации Catalyst/Tungsten:

- **Catalyst Optimizer:** логическая и физическая оптимизация планов
- **Tungsten:** бинарный формат, off-heap память, code generation
- **Predicate pushdown:** фильтры применяются как можно раньше
- **Broadcast join:** маленькие таблицы рассылаются всем executor'ам
- **Whole-stage codegen:** генерация оптимизированного байт-кода

In [ ]:
spark.stop()
print("✅ Spark сессия завершена")